# Forks: write-audit-publish (Python async)

Same scenario as `forks.ipynb` but built against
`uni_db.AsyncUni` so every database call is awaitable.
Useful for callers that already live in an asyncio event
loop (web servers, async agents).

Jupyter's IPython kernel runs cells inside an event loop,
so we can `await` directly — no `asyncio.run(...)` needed
at the cell level. The companion `run_notebooks.py` test
harness uses `PyCF_ALLOW_TOP_LEVEL_AWAIT` to execute the
same code outside Jupyter.


In [ ]:
import uni_db

## 1. Open a database, define schema

`uni_db.AsyncUni.builder()` is the async-API entry point
— every subsequent call returns a coroutine. Builders
are constructed synchronously and only their terminal
`.build()` / `.apply()` are awaitable.


In [ ]:
db = await uni_db.AsyncUni.builder().disable_fork_sweeper(True).build()

await db.schema().label("Person").property("name", "string").apply()
await db.schema().edge_type("KNOWS", ["Person"], ["Person"]).apply()
await db.schema().edge_type("FRIEND_OF_FRIEND", ["Person"], ["Person"]).apply()

print("schema applied")

## 2. Seed primary with a small social graph


In [ ]:
primary = db.session()
tx = await primary.tx()
await tx.execute(
    "CREATE (alice:Person {name: 'alice'}),"
    "       (bob:Person {name: 'bob'}),"
    "       (carol:Person {name: 'carol'}),"
    "       (dave:Person {name: 'dave'}),"
    "       (alice)-[:KNOWS]->(bob),"
    "       (bob)-[:KNOWS]->(carol),"
    "       (carol)-[:KNOWS]->(dave)"
)
await tx.commit()
await db.flush()

print("seeded primary with 4 vertices + 3 KNOWS edges")

## 3. Stage a hypothesis on a fork


In [ ]:
fork = await primary.fork("hypothesis_a").build()
tx = await fork.tx()
await tx.execute(
    "MATCH (a:Person)-[:KNOWS]->(b:Person)-[:KNOWS]->(c:Person) "
    "WHERE id(a) <> id(c) "
    "CREATE (a)-[:FRIEND_OF_FRIEND]->(c)"
)
await tx.commit()
await fork.flush()
del fork  # release the session before promote opens a fresh one

print("hypothesis_a: staged 2-hop FRIEND_OF_FRIEND derivation")

## 4. Audit via `diff_fork_primary`


In [ ]:
diff = await db.diff_fork_primary("hypothesis_a")

print(
    f"diff: +{len(diff.vertices.added)} vertices, "
    f"+{len(diff.edges.added)} edges, "
    f"~{len(diff.vertices.changed)} changed"
)

for e in diff.edges.added:
    short = e.edge_uid[-8:]
    print(f"  + (…{short}) [:{e.edge_type}] (between two Persons)")

## 5. Publish: `promote_from_fork`


In [ ]:
report = await db.promote_from_fork(
    "hypothesis_a",
    [
        uni_db.PromotePattern.label("Person"),
        uni_db.PromotePattern.edge_type("FRIEND_OF_FRIEND"),
    ],
)

print(
    f"inserted: {report.vertices_inserted} vertices, "
    f"{report.edges_inserted} edges "
    f"(skipped: {report.vertices_skipped_uid_conflict} UID conflicts, "
    f"{report.edges_skipped_duplicate} dup edges, "
    f"{report.edges_skipped_no_endpoint} orphans)"
)

## 6. Drop the fork; primary now has the new edges


In [ ]:
await db.drop_fork("hypothesis_a")

result = await primary.query(
    "MATCH (a:Person)-[r]->(b:Person) "
    "RETURN a.name AS src, type(r) AS rel, b.name AS dst "
    "ORDER BY rel, src, dst"
)

for row in result.rows:
    print(f"  ({row.get('src')})-[:{row.get('rel')}]->({row.get('dst')})")

## Wrap-up

The async API mirrors the sync surface 1:1 — every
method returns a coroutine, builders construct
synchronously and await their terminal step. From an
async caller you get the full fork track without the
sync runtime blocking your event loop.

Companions: `forks.ipynb` (Python sync) and
`examples/rust/forks.ipynb` (Rust via evcxr).
